In [ ]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

In [ ]:
# Your node data from Phase 1 & 2
nodes_data = {
    'node': [
        'atrium', 'mosque', 'rcdg', 'kalimudan', 'soccer_field', 'ebl',
        'library', 'carim', 'dorm2', 'csm', 'diwa', 'himati_house',
        'usc_house', 'castillo', 'ate_lings', 'lydia', 'aquatics',
        'dhk_training_gym', 'dc_sports_complex', 'cultural_center', 'som'
    ],
    'est_peak_pop': [350, 50, 350, 50, 50, 250, 100, 200, 10, 350, 50, 50, 50, 50, 50, 50, 50, 350, 100, 50, 350],
    'activity_type': [
        'sedentary', 'sedentary', 'vigorous', 'moderate', 'vigorous', 'moderate',
        'sedentary', 'sedentary', 'moderate', 'sedentary', 'sedentary', 'sedentary',
        'sedentary', 'moderate', 'sedentary', 'moderate', 'vigorous', 'vigorous',
        'vigorous', 'sedentary', 'sedentary'
    ]
}

# Define activity factors only (age factors removed)
activity_factors = {
    'sedentary': 1.0,
    'moderate': 1.3,
    'vigorous': 2.0
}

# Create DataFrame
df = pd.DataFrame(nodes_data)

In [ ]:
# Calculate Raw Risk (only using population and activity factor)
df['activity_factor'] = df['activity_type'].map(activity_factors)
df['raw_risk'] = df['est_peak_pop'] * df['activity_factor']

# Calculate total raw risk
total_raw_risk = df['raw_risk'].sum()

# Calculate normalized risk weight
df['normalized_risk_weight'] = df['raw_risk'] / total_raw_risk
df['risk_percentage'] = df['normalized_risk_weight'] * 100

# Calculate cumulative risk (for prioritizing nodes)
df_sorted = df.sort_values('normalized_risk_weight', ascending=False)
df_sorted['cumulative_risk'] = df_sorted['normalized_risk_weight'].cumsum()
df = df_sorted.sort_index()  # Re-sort back to original order

# Display results in console
print("=" * 90)
print("PHASE 3: RISK SCORE CALCULATION")
print("=" * 90)
print()
print(f"Total Raw Risk (sum): {total_raw_risk:.1f}")
print(f"Sum of Normalized Weights: {df['normalized_risk_weight'].sum():.4f}")
print("=" * 90)

PHASE 3: RISK SCORE CALCULATION

Total Raw Risk (sum): 3983.0
Sum of Normalized Weights: 1.0000


In [ ]:
output_file = 'phase3_risk_analysis.xlsx'

print(f"\n✅ Excel file exported successfully: '{output_file}'")
print("\n📊 File contains 4 sheets:")
print("   1. Risk_Calculation - Complete risk calculation for all nodes")
print("   2. Ranked_by_Risk - Nodes sorted from highest to lowest risk")
print("   3. Summary - Key statistics and insights")

# Print ranked nodes for quick reference
print("\n" + "=" * 90)
print("ALL NODES RANKED BY RISK (highest to lowest)")
print("=" * 90)
df_all = df.sort_values('normalized_risk_weight', ascending=False)
for idx, row in df_all.iterrows():
    print(f"{row['node']:20} | Risk: {row['risk_percentage']:5.2f}% | Population: {row['est_peak_pop']:3.0f} | {row['activity_type']}")


✅ Excel file exported successfully: 'phase3_risk_analysis.xlsx'

📊 File contains 4 sheets:
   1. Risk_Calculation - Complete risk calculation for all nodes
   2. Ranked_by_Risk - Nodes sorted from highest to lowest risk
   3. Summary - Key statistics and insights

ALL NODES RANKED BY RISK (highest to lowest)
rcdg                 | Risk: 17.57% | Population: 350 | vigorous
dhk_training_gym     | Risk: 17.57% | Population: 350 | vigorous
atrium               | Risk:  8.79% | Population: 350 | sedentary
csm                  | Risk:  8.79% | Population: 350 | sedentary
som                  | Risk:  8.79% | Population: 350 | sedentary
ebl                  | Risk:  8.16% | Population: 250 | moderate
carim                | Risk:  5.02% | Population: 200 | sedentary
dc_sports_complex    | Risk:  5.02% | Population: 100 | vigorous
soccer_field         | Risk:  2.51% | Population:  50 | vigorous
aquatics             | Risk:  2.51% | Population:  50 | vigorous
library              | Risk:  2.51%

In [ ]:
# Export to Excel with multiple sheets
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Complete risk calculation (without age columns)
    df_display = df[['node', 'est_peak_pop', 'activity_type', 'activity_factor',
                      'raw_risk', 'normalized_risk_weight', 'risk_percentage']].copy()
    df_display.to_excel(writer, sheet_name='Risk_Calculation', index=False)

    # Sheet 2: Nodes ranked by risk (highest to lowest)
    df_ranked = df.sort_values('normalized_risk_weight', ascending=False).copy()
    df_ranked['rank'] = range(1, len(df_ranked) + 1)
    df_ranked['cumulative_percentage'] = df_ranked['normalized_risk_weight'].cumsum() * 100
    df_ranked[['rank', 'node', 'raw_risk', 'normalized_risk_weight',
                'risk_percentage', 'cumulative_percentage']].to_excel(
        writer, sheet_name='Ranked_by_Risk', index=False
    )

    # Sheet 3: Summary statistics
    summary_data = {
        'Metric': [
            'Total Nodes',
            'Total Raw Risk',
            'Highest Risk Node',
            'Highest Risk Value',
            'Highest Risk %',
            'Lowest Risk Node',
            'Lowest Risk Value',
            'Lowest Risk %',
            'Top 3 Risk Cumulative %',
            'Top 5 Risk Cumulative %'
        ],
        'Value': [
            len(df),
            f"{total_raw_risk:.1f}",
            df_ranked.iloc[0]['node'],
            f"{df_ranked.iloc[0]['raw_risk']:.1f}",
            f"{df_ranked.iloc[0]['risk_percentage']:.2f}%",
            df_ranked.iloc[-1]['node'],
            f"{df_ranked.iloc[-1]['raw_risk']:.1f}",
            f"{df_ranked.iloc[-1]['risk_percentage']:.2f}%",
            f"{df_ranked.head(3)['risk_percentage'].sum():.2f}%",
            f"{df_ranked.head(5)['risk_percentage'].sum():.2f}%"
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

    # Sheet 4: Risk weights only (ready for Phase 4)
    risk_weights_df = df[['node', 'normalized_risk_weight']].copy()
    risk_weights_df.to_excel(writer, sheet_name='Risk_Weights_Phase4', index=False)

# Format the Excel file with colors and styling
wb = load_workbook(output_file)

# Style for Risk_Calculation sheet
ws = wb['Risk_Calculation']
# Add header formatting
header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# Add conditional formatting for risk percentages
for row in range(2, len(df) + 2):
    cell = ws.cell(row=row, column=7)  # risk_percentage column (adjusted index)
    risk_val = cell.value
    if risk_val >= 10:
        cell.fill = PatternFill(start_color="FF6B6B", end_color="FF6B6B", fill_type="solid")  # High risk - red
    elif risk_val >= 5:
        cell.fill = PatternFill(start_color="FFE66D", end_color="FFE66D", fill_type="solid")  # Medium risk - yellow
    else:
        cell.fill = PatternFill(start_color="C8E6C9", end_color="C8E6C9", fill_type="solid")  # Low risk - green

# Adjust column widths
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 30)
    ws.column_dimensions[col_letter].width = adjusted_width

# Style for Ranked_by_Risk sheet
ws2 = wb['Ranked_by_Risk']
for cell in ws2[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# Style for Summary sheet
ws3 = wb['Summary']
for cell in ws3[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

wb.save(output_file)